# theoryminer — tutorial notebook

[Open this notebook in Google Colab](https://colab.research.google.com/github/rasoulnorouzi/theoryminer/blob/main/tutorials/TUTORIAL.ipynb)

This notebook is the tutorial and the reference of `theoryminer`. Each section shows one function,
its settings, its output shape, and the printed summary. Section 1.5 finds the DOI of each paper with
`extract_dois`. Section 10 runs the whole pipeline on the sample papers, DOIs included. It runs in Google Colab, from `tutorials/`, or from the repository root. The examples use the open-access papers in
`sample_files/` (see `sample_files/SOURCES.md`). The slow step (`causenet` on all sample
sentences, a few minutes on CPU) loads a saved file when one exists.

The pipeline:

```
PDF ──extract_dois──► the DOI of each paper (a list, or a CSV file)
 │
 └──harvest──► sentences ──causenet──► relations (cause span → effect span)
                                              │
                    ┌─────────────────────────┼─────────────────────────┐
             standardize_constructs    pairwise_grouping            clusterer
             span → taxonomy concept   span → group (similarity)   span → cluster (density)
                    │                        └──── standardize_groups ────┘
                    │                               group → taxonomy concept
                    └─────────────────────────┬─────────────────────────┘
                                              ▼
                                  (next stage) theorize → directed graph
```

No function writes a result file. Keep the returned data in a variable. Write a file yourself
when you need one (section 8).

## 0. Setup

Run the next cell first. It does three things:

1. It changes to the repository root when the notebook runs inside `tutorials/`.
2. It installs `theoryminer` from PyPI with pip when the package is missing. This is the case in a
   new Google Colab session.
3. It downloads the sample papers from GitHub when the folder `sample_files/` is missing.

In a clone of the repository with `pip install -e .`, the cell installs and downloads nothing.
In Colab, a GPU runtime makes `causenet` and the embeddings faster. A CPU runtime also works.

In [1]:
import importlib
import json
import os
import subprocess
import sys
import urllib.request

REPO = "rasoulnorouzi/theoryminer"
PIP_TARGET = "theoryminer[umap]"          # the latest release on PyPI


def download_sample_files(folder="sample_files"):
    """Download every file of sample_files/ from GitHub into the folder."""
    listing_url = f"https://api.github.com/repos/{REPO}/contents/sample_files"
    with urllib.request.urlopen(listing_url) as response:
        listing = json.load(response)
    os.makedirs(folder, exist_ok=True)
    for item in listing:
        urllib.request.urlretrieve(item["download_url"], os.path.join(folder, item["name"]))
    print(f"downloaded {len(listing)} files into {folder}/")


# 1. Jupyter starts a notebook in its own folder. The paths below start at the repository root.
if os.path.basename(os.getcwd()) == "tutorials":
    os.chdir("..")

# 2. Install the package when pip has no record of it, as in a new Colab session.
# Ask pip's records, not the import system: an old cache folder alone can look like an
# empty "theoryminer" package, and an import would keep that empty package in memory.
try:
    importlib.metadata.version("theoryminer")
except importlib.metadata.PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIP_TARGET], check=True)
    importlib.invalidate_caches()

# 3. Get the sample papers when this folder is not a clone of the repository.
if not os.path.isdir("sample_files"):
    download_sample_files()

print("working folder:", os.getcwd())

working folder: /home/rass/Desktop/projects/theoryminer


In [2]:
import collections, json, os

from theoryminer import harvest, causenet, DEFAULT_STEPS, STEPS
from theoryminer.harmonizer import (
    standardize_constructs, standardize_groups, pairwise_grouping, clusterer, label_groups)

SAMPLE = "sample_files"                     # a folder of PDF files
ONE_PDF = os.path.join(SAMPLE, sorted(n for n in os.listdir(SAMPLE) if n.endswith(".pdf"))[0])
os.makedirs("outputs", exist_ok=True)
print("one PDF:", ONE_PDF)

one PDF: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf


## 1. `harvest` — PDF to clean sentences

```python
data = harvest(source, pages=None, skip_pages=None, steps=None, min_words=4, max_words=80, cache=True)
```

`source` is one PDF file or a folder of PDF files. The function returns one dict with two lists:
`sentences` (kept rows) and `dropped` (the drop log). Each kept row has `doc_id`, `page`,
`sent_id`, `section`, `raw`, `clean`, `flag`. Each dropped row has `drop_reason`. The first call
extracts the page text with `pdf_inspector` and saves it next to each PDF as `<pdf>.pages.json`.
Later calls are fast.

In [3]:
data = harvest(SAMPLE)
print()
print("keys:", list(data))
print("first kept row:", data["sentences"][0])

  reading .pdf file 6 of 7

  reading .pdf file 7 of 7
harvest: sample_files
  files: 8 | read: 7 | skipped: 1
    skipped: SOURCES.md: the file name does not end with .pdf (not_pdf)
  text: 2,547 sentences kept | 1,849 rows dropped
       423  table_row
       328  no_lowercase
       247  reference_section
       229  too_short
       148  low_alpha
       107  running_head
       106  heading
        74  digit_heavy
        72  spaced_letters
        52  caption_or_keywords
        31  ref_fragment
        13  index_entry
         8  doi_url
         7  too_long
         2  empty_page
         1  page_number
         1  front_back_matter
  sentences per file:
       265  alrabai_2021_autonomy_supportive_teaching
       297  barrech_2018_job_insecurity_health
       301  friesinger_2025_discrimination_health
       385  gordesli_2024_social_media_mental_health
       318  pelikan_2021_needs_intrinsic_motivation
       616  schutz_2025_loneliness_social_support
       365  stephany_2017_income_inequality_trust



### 1.0 One file or a folder

For a folder, `harvest` reads each PDF on its own and then joins the results. The steps never
mix two papers. Each `sent_id` starts with the document number: `d002p0014s001` is document 2,
paragraph 14, sentence 1. So a `sent_id` is unique in the whole folder.

A bad file does not stop a folder run. It gets one row in the drop log with `page` 0:
`not_pdf` for a file whose name does not end with `.pdf`, `unreadable_pdf` for a PDF that
`pdf_inspector` cannot read or that has no text layer. `SOURCES.md` in the sample folder shows
the first case. For one PDF, a bad file raises a `ValueError`.

In [4]:
skipped = [r for r in data["dropped"] if r["page"] == 0]
print("skipped files:", skipped)
print()
single = harvest(ONE_PDF)
print()
print("sent_id of one PDF:", single["sentences"][0]["sent_id"])
print("doc_ids in the folder run:", sorted({r["doc_id"] for r in data["sentences"]}))

skipped files: [{'doc_id': 'SOURCES', 'page': 0, 'section': '', 'drop_reason': 'not_pdf', 'text': 'SOURCES.md: the file name does not end with .pdf'}]

harvest: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf
  files: 1 | read: 1 | skipped: 0
  text: 265 sentences kept | 153 rows dropped
        47  reference_section
        34  running_head
        30  table_row
        23  heading
         8  too_short
         4  caption_or_keywords
         4  too_long
         2  no_lowercase
         1  digit_heavy

sent_id of one PDF: d000p0002s000
doc_ids in the folder run: ['alrabai_2021_autonomy_supportive_teaching', 'barrech_2018_job_insecurity_health', 'friesinger_2025_discrimination_health', 'gordesli_2024_social_media_mental_health', 'pelikan_2021_needs_intrinsic_motivation', 'schutz_2025_loneliness_social_support', 'stephany_2017_income_inequality_trust']


### 1.1 What `strip_citations` and `strip_stats` do: `raw` against `clean`

In [5]:
shown = 0
for r in data["sentences"]:
    if r["raw"] != r["clean"] and 60 < len(r["raw"]) < 260:
        print("RAW  :", r["raw"]); print("CLEAN:", r["clean"]); print()
        shown += 1
    if shown == 3: break

RAW  : In another definition, Benson (2001) perceived autonomy as the capacity to take charge of one's own leaning based on his desire, ability, and degree of freedom.
CLEAN: In another definition, Benson perceived autonomy as the capacity to take charge of one's own leaning based on his desire, ability, and degree of freedom.

RAW  : Autonomy-supportive teaching, according to Reeve (2016), is the delivery of instruction through an interpersonal tone of understanding that appreciates, supports, and vitalizes students' psychological needs.
CLEAN: Autonomy-supportive teaching, according to Reeve, is the delivery of instruction through an interpersonal tone of understanding that appreciates, supports, and vitalizes students' psychological needs.

RAW  : In this regard, Ikonen (2013) emphasized that the movement toward autonomy-supportive teaching in institutional settings has proven to fit surprisingly well.
CLEAN: In this regard, Ikonen emphasized that the movement toward autonomy-suppor

### 1.2 Drop reasons

Each dropped row carries the rule that removed it. The drop log sets the recall denominator.

In [6]:
reasons = collections.Counter(r["drop_reason"] for r in data["dropped"])
for reason, n in reasons.most_common():
    print(f"  {n:>6}  {reason}")
print()
flags = collections.Counter(r["flag"] for r in data["sentences"])
print("flags on kept rows:", dict(flags))

     423  table_row
     328  no_lowercase
     247  reference_section
     229  too_short
     148  low_alpha
     107  running_head
     106  heading
      74  digit_heavy
      72  spaced_letters
      52  caption_or_keywords
      31  ref_fragment
      13  index_entry
       8  doi_url
       7  too_long
       2  empty_page
       1  not_pdf
       1  page_number
       1  front_back_matter

flags on kept rows: {'fragment': 167, '': 2367, 'front_matter': 13}


### 1.2b Flags, and how to leave flagged sentences out

A flag marks a doubtful sentence. The sentence stays in `sentences`. The flags are:

| flag | meaning |
|---|---|
| `front_matter` | the PDF extractor mixed journal sidebar text into the sentence: an editor name, a date, an e-mail address |
| `fragment` | the sentence has no final punctuation |
| `long` | the sentence has more than 80 words |

To keep a flagged sentence away from the model, filter the list before `causenet`:

In [7]:
front = [s for s in data["sentences"] if s["flag"] == "front_matter"]
print(len(front), "sentences flagged front_matter; the first one:")
print("  ", front[0]["clean"][:160])

without_front_matter = [s for s in data["sentences"] if s["flag"] != "front_matter"]
print(len(data["sentences"]), "->", len(without_front_matter), "sentences")

13 sentences flagged front_matter; the first one:
   Strategies targeting learners' self-determined learning in the classroom in terms of satisfying learner basic psychological needs (BPNs) of autonomy, competence
2547 -> 2534 sentences


### 1.3 The steps list

Cleaning is an ordered list of steps. Remove a name, change the order, or insert your own
function. The contract for each step: take a list of row dicts, return a list of row dicts. To
drop a row, set `kept=False` and a `drop_reason`. Do not delete the row.

In [8]:
print(DEFAULT_STEPS)

def no_questions(rows):
    """Drop sentences that end with a question mark."""
    for r in rows:
        if r["kept"] and r["text"].rstrip().endswith("?"):
            r["kept"] = False
            r["drop_reason"] = "question"
    return rows

custom = harvest(ONE_PDF,
                 steps=["fix_unicode", "strip_markdown", "drop_junk_lines", "join_paragraphs",
                        "split_sentences", no_questions, "filter_length"])
print()
print("dropped as question:", [r["text"] for r in custom["dropped"] if r["drop_reason"] == "question"][:3])

['fix_unicode', 'strip_markdown', 'drop_running_heads', 'drop_junk_lines', 'join_paragraphs', 'split_sentences', 'strip_citations', 'strip_stats', 'drop_junk_sentences', 'filter_length']
harvest: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf
  files: 1 | read: 1 | skipped: 0
  text: 295 sentences kept | 122 rows dropped
        50  reference_section
        30  table_row
        23  heading
        10  too_short
         6  too_long
         3  question

dropped as question: ['RQ1: the autonomy of Saudi EFL learners?', "What are the most important strategies to promote RQ2: enhance EFL learner's autonomy?", 'Will teacher autonomy-supportive teaching RQ2: would causally predict learner EFL autonomy?']


### 1.4 Page control

`pages` keeps a window. `skip_pages` removes pages inside it.

In [9]:
small = harvest(ONE_PDF, pages="2-6", skip_pages=[4])
print()
print("pages seen:", sorted({r["page"] for r in small["sentences"]}))

harvest: sample_files/alrabai_2021_autonomy_supportive_teaching.pdf
  files: 1 | read: 1 | skipped: 0
  text: 82 sentences kept | 26 rows dropped
        11  table_row
         5  heading
         4  running_head
         3  too_long
         1  digit_heavy
         1  too_short
         1  caption_or_keywords

pages seen: [2, 3, 5, 6]


### 1.5 The DOI of each paper: `extract_dois`

```python
rows = extract_dois(source, save=None, first_pages=2, cache=True)
```

`source` is one PDF or a folder, as in `harvest`. A paper holds many DOIs, most of them in its
reference list. The DOI of the paper itself is at the top of page 1: in the header, the citation box
or the footer. So the rule is: the first DOI on page 1, else the first DOI on page 2. It was checked by
hand on the 7 sample papers: 7 of 7 correct.

The result is one row per file: `doc_id`, `file`, `doi`, `page`, `status`, `note`. The `status` is
`found`, `not_found`, `not_pdf` or `unreadable_pdf`. A bad file does not stop the run.
`save="outputs"` writes `outputs/dois.csv`. `save="outputs/my_dois.csv"` writes that file.

In [3]:
from theoryminer import extract_dois

rows = extract_dois(SAMPLE, save="outputs")
print()
for row in rows:
    print(f"  {row['status']:<10} {row['doi']:<30} {row['file']}")

  reading .pdf file 7 of 7
dois: sample_files
  files: 8 | found: 7 | not found: 0 | skipped: 1
    not_pdf: SOURCES.md: the file name does not end with .pdf
  saved: outputs/dois.csv

  not_pdf                                   SOURCES.md
  found      10.3389/fpsyg.2021.728657      alrabai_2021_autonomy_supportive_teaching.pdf
  found      10.1186/s12889-018-5621-4      barrech_2018_job_insecurity_health.pdf
  found      10.1186/s12889-025-23888-6     friesinger_2025_discrimination_health.pdf
  found      10.1371/journal.pone.0316365   gordesli_2024_social_media_mental_health.pdf
  found      10.1371/journal.pone.0257346   pelikan_2021_needs_intrinsic_motivation.pdf
  found      10.1186/s12889-025-23247-5     schutz_2025_loneliness_social_support.pdf
  found      10.1007/s11205-016-1460-9      stephany_2017_income_inequality_trust.pdf


## 2. `causenet` — sentences to cause-effect pairs

```python
relations = causenet(sentences, threshold=0.8, mode="neural", decision="cls+span", batch_size=64,
                     device="auto")
```

The model is `rasoultilburg/SocioCausaNet`. It loads once per session. `device="auto"` runs it on
an NVIDIA GPU when torch can see one, else on an Apple Silicon GPU, else on the CPU. Give `"cpu"`,
`"cuda"` or `"mps"` to choose. On a
GPU, a larger `batch_size` is usually faster. It returns one dict per
relation: `rel_id`, `sent_id`, `doc_id`, `page`, `cause`, `effect`, `sentence`. `rel_id` is
`sent_id` + `r<n>`, the n-th relation of the sentence. The spans are substrings of the sentence.
Nothing paraphrases them. A cut span such as `'teachers ( i.'` stays as it is: the span is the
provenance key.

In [11]:
causenet(["Smoking causes lung cancer and heart disease",
          "Exercise improves physical health and mental well-being",
          "This is a non-causal sentence about weather"])

/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


causenet: loading rasoultilburg/SocioCausaNet ...


Loading weights:   0%|          | 0/217 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 217/217 [00:00<00:00, 12611.21it/s]

causenet: device = cpu
  3/3 sentences (0s)
causenet: 2 causal sentences of 3 | 4 relations
  spans repaired after decoding: 0 | both spans found verbatim in their sentence: 100%


[{'rel_id': 's0000r00',
  'sent_id': 's0000',
  'doc_id': 'input',
  'page': 0,
  'cause': 'smoking',
  'effect': 'lung cancer',
  'sentence': 'Smoking causes lung cancer and heart disease'},
 {'rel_id': 's0000r01',
  'sent_id': 's0000',
  'doc_id': 'input',
  'page': 0,
  'cause': 'smoking',
  'effect': 'heart disease',
  'sentence': 'Smoking causes lung cancer and heart disease'},
 {'rel_id': 's0001r00',
  'sent_id': 's0001',
  'doc_id': 'input',
  'page': 0,
  'cause': 'exercise',
  'effect': 'physical health',
  'sentence': 'Exercise improves physical health and mental well-being'},
 {'rel_id': 's0001r01',
  'sent_id': 's0001',
  'doc_id': 'input',
  'page': 0,
  'cause': 'exercise',
  'effect': 'mental well-being',
  'sentence': 'Exercise improves physical health and mental well-being'}]

### 2.1 A real run: all sample papers

A few minutes on CPU. The cell loads the saved file when it exists.

In [12]:
REL_FILE = "outputs/relations_sample.json"
if os.path.exists(REL_FILE):
    relations = json.load(open(REL_FILE))
    print(f"loaded {len(relations):,} relations from {REL_FILE}")
else:
    relations = causenet(data["sentences"])
    json.dump(relations, open(REL_FILE, "w"))

print()
print("relations[0]:", relations[0])
print()
for r in relations[:6]:
    print(f"  [{r['doc_id'][:20]} p{r['page']} {r['rel_id']}] {r['cause']!r} -> {r['effect']!r}")

loaded 979 relations from outputs/relations_sample.json

relations[0]: {'rel_id': 'd000p0003s003r00', 'sent_id': 'd000p0003s003', 'doc_id': 'alrabai_2021_autonomy_supportive_teaching', 'page': 1, 'cause': 'experimental intervention', 'effect': 'efl ehsan namaziandost', 'sentence': 'The Reviewed by: findings derived out from analyses of variance, covariance and a hierarchical regression Zhengdong Gan, University of Macau, China revealed that the experimental intervention led to statistically significant increased EFL Ehsan Namaziandost, autonomy for learners in the experimental group.'}

  [alrabai_2021_autonom p1 d000p0003s003r00] 'experimental intervention' -> 'efl ehsan namaziandost'
  [alrabai_2021_autonom p1 d000p0006s003r00] 'desire' -> 'autonomy'
  [alrabai_2021_autonom p1 d000p0006s003r01] 'desire' -> 'take charge'
  [alrabai_2021_autonom p1 d000p0006s003r02] 'desire' -> 's own leaning'
  [alrabai_2021_autonom p1 d000p0006s003r03] 'ability' -> 'autonomy'
  [alrabai_2021_autonom 

### 2.2 The threshold

A lower threshold gives more pairs. The old pipeline swept 0.5–0.8 and adopted 0.8.

In [13]:
sample = [r["sentence"] for r in relations[:40]]
strict = causenet(sample, threshold=0.8)
loose  = causenet(sample, threshold=0.5)
print()
print(f"threshold 0.8: {len(strict)} relations | threshold 0.5: {len(loose)} relations")

  40/40 sentences (3s)
causenet: 40 causal sentences of 40 | 242 relations
  spans repaired after decoding: 66 | both spans found verbatim in their sentence: 99%


  40/40 sentences (2s)
causenet: 40 causal sentences of 40 | 268 relations
  spans repaired after decoding: 89 | both spans found verbatim in their sentence: 98%

threshold 0.8: 242 relations | threshold 0.5: 268 relations


### 2.3 One construct, two roles

A span that is a cause in one sentence and an effect in another is one node. Chains pass through these spans only.

In [14]:
causes  = collections.Counter(r["cause"] for r in relations)
effects = collections.Counter(r["effect"] for r in relations)
both = sorted((s for s in causes if s in effects), key=lambda s: -(causes[s] + effects[s]))
print(f"{len(causes):,} distinct causes | {len(effects):,} distinct effects | {len(both)} spans in both roles")
for s in both[:6]:
    print(f"  {s!r}: cause x{causes[s]}, effect x{effects[s]}")
s = both[0]
print()
print("as CAUSE :", next(f"[{r['rel_id']}] {r['sentence']}" for r in relations if r["cause"] == s))
print("as EFFECT:", next(f"[{r['rel_id']}] {r['sentence']}" for r in relations if r["effect"] == s))

572 distinct causes | 694 distinct effects | 47 spans in both roles
  'this': cause x18, effect x4
  'social support': cause x20, effect x2
  'loneliness': cause x15, effect x6
  'job insecurity': cause x8, effect x4
  'autonomy': cause x4, effect x4
  'health': cause x1, effect x6

as CAUSE : [d002p0030s003r00] This criterion may have excluded recent movers to Agder County and newly arrived immigrants to Norway.
as EFFECT: [d000p0023s002r00] This is because ANOVA and ANCOVA analyses are powerful enough to disclose the net effect size of the treatment manipulated in this study.


## 3. The harmonizer — one idea, three routes

Harmonization decides which spans are the **same node**. All three routes take the `relations`
list (or a plain list of strings), pool causes and effects into one set of unique spans, and
return a dict keyed by the original span. Vectors are L2-normalized in one place, so each
similarity is a cosine.

| route | function | node identity by |
|---|---|---|
| taxonomy | `standardize_constructs` | closeness to a thesaurus concept |
| pairwise | `pairwise_grouping` | closeness to other spans (threshold, connected components) |
| density | `clusterer` | HDBSCAN cluster |
| taxonomy for groups | `standardize_groups` | closeness of a whole group to a concept |

Models: `allmpnet` (default), `bge_base`, `minilm`, or any sentence-transformers id.

`tidy_span` repairs what the tokenizer breaks when the model decodes a span. `causenet`
applies it once. It removes no word. No function removes function words: the models handle
them, and words such as *lack*, *not*, *low* carry the polarity of a construct.

In [15]:
from theoryminer.harmonizer.spans import collect_spans, tidy_span
spans = collect_spans(relations)
print(len(spans), "unique spans; first five:", spans[:5])
for q in ["society ' s desired values", "[CLS] autonomy support is critical", "inte ##grative processes",
          "), and intrapersonal events ( self-set goals )", "intrinsic motivation,", "the lack of autonomy"]:
    print(f"  tidy_span({q!r}) -> {tidy_span(q)!r}")
verbatim = sum(r["cause"].lower() in r["sentence"].lower() and r["effect"].lower() in r["sentence"].lower()
               for r in relations)
print(f"both spans found verbatim in their sentence: {verbatim / len(relations):.0%}")

1219 unique spans; first five: ['experimental intervention', 'efl ehsan namaziandost', 'desire', 'autonomy', 'take charge']
  tidy_span("society ' s desired values") -> "society's desired values"
  tidy_span('[CLS] autonomy support is critical') -> 'autonomy support is critical'
  tidy_span('inte ##grative processes') -> 'integrative processes'
  tidy_span('), and intrapersonal events ( self-set goals )') -> 'and intrapersonal events (self-set goals)'
  tidy_span('intrinsic motivation,') -> 'intrinsic motivation'
  tidy_span('the lack of autonomy') -> 'the lack of autonomy'
both spans found verbatim in their sentence: 97%


## 4. `standardize_constructs` — spans to taxonomy concepts

```python
matches = standardize_constructs(relations, taxonomy="elsst", strategy="enriched", top=5,
                                 embeddings="allmpnet", leaves_only=True, cache=True)
```

Returns `{span: [{"id", "leaf", "path", "score"}, ...]}`, best first, one item per concept. No
threshold is applied. τ is a downstream decision.

In [16]:
matches = standardize_constructs(relations, strategy="enriched", top=5)
print()
for span in list(matches)[:6]:
    print(f"{span:<62}", " | ".join(f"{m['leaf']} ({m['score']:.2f})" for m in matches[span][:3]))
print()
print("full record for one span:", matches[list(matches)[0]][0])

taxonomy: 3,436 concepts | 2,417 leaves | 2,661 entries (leaves only)
taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)
embeddings: loading sentence-transformers/all-mpnet-base-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12005.62it/s]

standardize: 1,219 spans -> top 5 of 2,661 entries | best score: median 0.49, min 0.16, max 0.80

experimental intervention                                      PSYCHOLOGICAL TESTS (0.44) | DEMONSTRATIONS (LESSONS) (0.40) | ANALYSIS (0.35)
efl ehsan namaziandost                                         ELSST Thesaurus (0.25) | NEO-NAZISM (0.25) | EXAMINATIONS (0.24)
desire                                                         ASPIRATION (0.43) | INCENTIVES (0.39) | BOREDOM (0.39)
autonomy                                                       AUTONOMY AT WORK (0.52) | INDIVIDUALISM (0.41) | CLASS CONSCIOUSNESS (0.39)
take charge                                                    POLITICAL ACTION (0.41) | OUTWORK (0.36) | LEADERSHIP (0.34)
s own leaning                                                  PREJUDICE (0.36) | ATTITUDE CHANGE (0.33) | SPOUSE'S RELIGIOUS BELIEF (0.32)

full record for one span: {'id': 'https://elsst.cessda.eu/id/5/9ecc4359-36f2-495a-a685-5bf729329efd', 'leaf': 

### 4.1 The score distribution

Read it before you choose τ. Many spans sit far from any ELSST concept. That is a fact about the thesaurus.

In [17]:
best = sorted(m[0]["score"] for m in matches.values())
n = len(best)
print(f"n={n:,}  p10={best[n//10]:.2f}  median={best[n//2]:.2f}  p90={best[9*n//10]:.2f}")
for tau in (0.50, 0.55, 0.60, 0.70):
    print(f"  share >= {tau}: {sum(b >= tau for b in best) / n:.1%}")
by_concept = collections.Counter(m[0]["leaf"] for m in matches.values())
print()
print(f"{n:,} spans -> {len(by_concept)} distinct top-1 concepts; largest:", by_concept.most_common(4))

n=1,219  p10=0.35  median=0.49  p90=0.63
  share >= 0.5: 47.4%
  share >= 0.55: 28.6%
  share >= 0.6: 14.8%
  share >= 0.7: 3.4%

1,219 spans -> 401 distinct top-1 concepts; largest: [('LONELINESS', 76), ('NERVOUS BREAKDOWN', 43), ('LEARNING', 39), ('INTERPERSONAL TRUST', 31)]


### 4.2 `leaves_only`: where a span may land

A **leaf** has no narrower concept. A **branch** has children. `leaves_only=True` (default) lets
a span land on leaves only. `leaves_only=False` adds the branches. The switch changes the set
of possible answers, not a score. `d` is the depth of the concept in the tree. Full
explanation: `tutorials/STRATEGIES.md`.

In [18]:
probe = ["autonomy", "intrinsic motivation", "well-being", "autonomy support",
         "loneliness", "unemployment", "social trust", "depressive symptoms"]
m_leaf = standardize_constructs(probe, leaves_only=True,  top=1)
m_all  = standardize_constructs(probe, leaves_only=False, top=1)
print()
for s in probe:
    a, b = m_leaf[s][0], m_all[s][0]
    print(f"  {s:<22} leaves: {a['leaf']:<22} d={a['path'].count('>') + 1} ({a['score']:.2f})   "
          f"all: {b['leaf']:<22} d={b['path'].count('>') + 1} ({b['score']:.2f})")

taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)
standardize: 8 spans -> top 1 of 2,661 entries | best score: median 0.60, min 0.46, max 0.69


taxonomy: 3,436 concepts | 2,417 leaves | 3,739 entries (all concepts)
taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)
standardize: 8 spans -> top 1 of 3,739 entries | best score: median 0.62, min 0.52, max 0.76

  autonomy               leaves: AUTONOMY AT WORK       d=3 (0.52)   all: AUTONOMY AT WORK       d=3 (0.52)
  intrinsic motivation   leaves: INCENTIVES             d=2 (0.65)   all: MOTIVATION             d=1 (0.69)
  well-being             leaves: HAPPINESS              d=3 (0.55)   all: WELL-BEING (HEALTH)    d=1 (0.61)
  autonomy support       leaves: AUTONOMY AT WORK       d=3 (0.57)   all: AUTONOMY AT WORK       d=3 (0.57)
  loneliness             leaves: LONELINESS             d=3 (0.63)   all: LONELINESS             d=3 (0.63)
  unemployment           leaves: UNEMPLOYED             d=3 (0.64)   all: UNEMPLOYMENT           d=2 (0.69)
  social trust           leaves: INTERPERSONAL TRUST    d=3 (0.69)   all: TRUST                  d=2 (0.76)
  depressive 

### 4.3 The six strategies

The strategy decides what text stands for a concept. `leaf` = the label. `path`, `anchor`,
`context`, `bracket` = the label with the tree in four phrasings. `enriched` = label, synonyms,
note, path. The winner rarely changes across strategies; the score does. A τ set for one
strategy is wrong for another.

In [19]:
from theoryminer.harmonizer.taxonomy import load_taxonomy, strategy_text, STRATEGIES
e = next(x for x in load_taxonomy("elsst", leaves_only=True) if x["leaf"] == "SELF-ESTEEM")
for s in STRATEGIES:
    print(f"  {s:<9} {strategy_text(e, s)}")
print()
for strategy in STRATEGIES:
    m = standardize_constructs(probe[:6], strategy=strategy, top=1)
    print(f"{strategy:<9}", " | ".join(f"{sp}: {m[sp][0]['leaf']} ({m[sp][0]['score']:.2f})" for sp in probe[:6]))

  leaf      self-esteem
  path      identity > personal identity > self-esteem
  anchor    self-esteem: identity > personal identity > self-esteem
  context   self-esteem is related to personal identity, identity
  bracket   self-esteem (personal identity, identity)
  enriched  self-esteem. self-acceptance; self-confidence; self-respect. identity > personal identity > self-esteem

taxonomy: embeddings from cache (elsst_leaf_allmpnet_leaves.npy)


standardize: 6 spans -> top 1 of 2,661 entries | best score: median 0.73, min 0.56, max 1.00
leaf      autonomy: AUTONOMY AT WORK (0.73) | intrinsic motivation: INCENTIVES (0.56) | well-being: MENTAL HEALTH (0.70) | autonomy support: AUTONOMY AT WORK (0.72) | loneliness: LONELINESS (1.00) | unemployment: UNEMPLOYED (0.77)
taxonomy: embeddings from cache (elsst_path_allmpnet_leaves.npy)


standardize: 6 spans -> top 1 of 2,661 entries | best score: median 0.61, min 0.57, max 0.72
path      autonomy: AUTONOMY AT WORK (0.57) | intrinsic motivation: INCENTIVES (0.72) | well-being: HAPPINESS (0.66) | autonomy support: AUTONOMY AT WORK (0.59) | loneliness: LONELINESS (0.61) | unemployment: UNEMPLOYED (0.62)
taxonomy: embeddings from cache (elsst_anchor_allmpnet_leaves.npy)
standardize: 6 spans -> top 1 of 2,661 entries | best score: median 0.63, min 0.62, max 0.67
anchor    autonomy: AUTONOMY AT WORK (0.64) | intrinsic motivation: INCENTIVES (0.62) | well-being: HAPPINESS (0.63) | autonomy support: AUTONOMY AT WORK (0.67) | loneliness: LONELINESS (0.62) | unemployment: UNEMPLOYED (0.66)
taxonomy: embeddings from cache (elsst_context_allmpnet_leaves.npy)


standardize: 6 spans -> top 1 of 2,661 entries | best score: median 0.62, min 0.55, max 0.66
context   autonomy: AUTONOMY AT WORK (0.66) | intrinsic motivation: INCENTIVES (0.62) | well-being: HAPPINESS (0.55) | autonomy support: AUTONOMY AT WORK (0.65) | loneliness: LONELINESS (0.62) | unemployment: UNEMPLOYED (0.61)
taxonomy: embeddings from cache (elsst_bracket_allmpnet_leaves.npy)
standardize: 6 spans -> top 1 of 2,661 entries | best score: median 0.68, min 0.63, max 0.70
bracket   autonomy: AUTONOMY AT WORK (0.63) | intrinsic motivation: INCENTIVES (0.70) | well-being: HAPPINESS (0.70) | autonomy support: AUTONOMY AT WORK (0.65) | loneliness: LONELINESS (0.64) | unemployment: UNEMPLOYED (0.70)
taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 6 spans -> top 1 of 2,661 entries | best score: median 0.60, min 0.52, max 0.65
enriched  autonomy: AUTONOMY AT WORK (0.52) | intrinsic motivation: INCENTIVES (0.65) | well-being: HAPPINESS (0.55) | autonomy support: AUTONOMY AT WORK (0.57) | loneliness: LONELINESS (0.63) | unemployment: UNEMPLOYED (0.64)


### 4.4 The cache

The first call for a (taxonomy, strategy, model, leaves/all) combination embeds all entries and
saves them in `theoryminer/taxonomy_cache/`. Files are plain: `.npy` vectors and a
`.json` entry list in the same row order. R reads them with `RcppCNPy` and `jsonlite`.

In [20]:
from theoryminer.harmonizer.taxonomy import CACHE_DIR
import numpy as np
files = sorted(os.listdir(CACHE_DIR))
print(len(files), "files, e.g.", files[:4])
v = np.load(os.path.join(CACHE_DIR, "elsst_enriched_allmpnet_leaves.npy"))
entries = json.load(open(os.path.join(CACHE_DIR, "elsst_leaves.json"), encoding="utf-8"))
print("vectors:", v.shape, v.dtype, "| entries:", len(entries), "| row 0 =", entries[0]["leaf"])

9 files, e.g. ['elsst_all.json', 'elsst_anchor_allmpnet_leaves.npy', 'elsst_bracket_allmpnet_leaves.npy', 'elsst_context_allmpnet_leaves.npy']
vectors: (2661, 768) float32 | entries: 2661 | row 0 = ELSST Thesaurus


## 5. `pairwise_grouping` — spans to groups by similarity

```python
groups = pairwise_grouping(relations, embeddings="allmpnet", threshold=0.69)
```

Two spans link when cosine ≥ threshold. A group is a connected component. `central` is the
medoid. 0.69 is Paper 3's calibrated threshold for all-mpnet.

In [21]:
groups = pairwise_grouping(relations, threshold=0.69)
print()
by_group = collections.defaultdict(list)
for span, g in groups.items():
    by_group[g["group"]].append(span)
sizes = sorted(by_group.values(), key=len, reverse=True)
print("largest group:", len(sizes[0]), "| singletons:", sum(len(m) == 1 for m in sizes))
print()
print("a good group:", next(m for m in sizes if 4 <= len(m) <= 6))
print()
print("one span:", groups[list(groups)[0]])

pairwise: 1,219 spans -> 669 groups at cos >= 0.69 | 90 groups with 2+ members, largest 153

largest group: 153 | singletons: 579

a good group: ['covid-19 pandemic', 'cope with the negative impacts of the pandemic', 'cope with the negative impact of the pandemic', 'covid-19', 'specificity of the covid-19 situation', 'pandemic']

one span: {'group': 0, 'central': 'experimental intervention'}


### 5.1 The chaining problem

Connected components are single linkage. Links chain. At 0.69 one component can hold a large
share of the spans. The cell finds one chain inside it: A links B, B links C, A and C are unrelated. When
the largest group is large, use `clusterer`, a higher threshold, or `standardize_constructs`.

In [22]:
from theoryminer.harmonizer.embeddings import embed
big = sizes[0]
print("largest component, sample:", big[:8])
V = embed(big); S = V @ V.T
best = None
for i in range(len(big)):
    for j in range(len(big)):
        if i == j or S[i, j] < 0.69: continue
        for k in range(len(big)):
            if k in (i, j) or S[j, k] < 0.69: continue
            if best is None or S[i, k] < best[0]:
                best = (S[i, k], big[i], big[j], big[k], S[i, j], S[j, k])
if best is None:
    print("no chain of three spans in the largest group")
else:
    print(f"chain: {best[1]!r} -{best[4]:.2f}- {best[2]!r} -{best[5]:.2f}- {best[3]!r}; direct cosine {best[0]:.2f}")
print()
for t in (0.60, 0.65, 0.69, 0.75, 0.80):
    pairwise_grouping(relations, threshold=t)

largest component, sample: ['relatedness', 'trust in the', 'experience discrimination', 'discrimination', 'reporting discrimination for one', 'individuals experiencing discrimination for three or more reasons', 'discrimination in more subtle ways', 'racism']


chain: 'social trust' -0.73- 'inequality and trust' -0.70- 'perception of inequality'; direct cosine 0.33



pairwise: 1,219 spans -> 421 groups at cos >= 0.6 | 67 groups with 2+ members, largest 670


pairwise: 1,219 spans -> 550 groups at cos >= 0.65 | 84 groups with 2+ members, largest 269


pairwise: 1,219 spans -> 669 groups at cos >= 0.69 | 90 groups with 2+ members, largest 153


pairwise: 1,219 spans -> 845 groups at cos >= 0.75 | 90 groups with 2+ members, largest 45


pairwise: 1,219 spans -> 983 groups at cos >= 0.8 | 78 groups with 2+ members, largest 34


## 6. `clusterer` — spans to density clusters

```python
clusters = clusterer(relations, embeddings="allmpnet", min_cluster_size=2, min_samples=2, umap=None)
```

HDBSCAN on the normalized vectors. Defaults are Paper 3's calibrated configuration. Group −1 is
noise: a span with no near neighbour. It becomes a node of one. It is never dropped.

In [23]:
clusters = clusterer(relations)
print()
by_cluster = collections.defaultdict(list)
for span, g in clusters.items():
    by_cluster[g["group"]].append(span)
for cid, members in sorted(by_cluster.items(), key=lambda kv: -len(kv[1]))[:6]:
    if cid == -1: continue
    print(f"C{cid} [{clusters[members[0]]['central']}]  {members[:5]} ...")
print()
print("noise example:", clusters[by_cluster[-1][0]])
print("noise share:", f"{len(by_cluster[-1]) / len(clusters):.0%}")

/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


cluster: 1,219 spans -> 244 clusters | 480 noise (39%) | raw embeddings

C214 [higher intrinsic motivation]  ['greater classroom motivation', "learner's intrinsic motivation was", 'intrinsic motivation', 'high intrinsic motivation', 'low intrinsic motivation'] ...
C79 [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful]  ["teachers'difficulties in adapting to remote teaching", 'distance learning', 'switched to distance learning', 'distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful', 'less structured learning environment in distance learning'] ...
C85 [perceived competence]  ['greater perceived competence', 'competence', "students'perceptions of competence", 'perceive themselves as competent', 'high perceived competence'] ...
C127 [procrastination]  ['maladaptive behavior like procrastination', 'procrastinated less', 'procrastinate less', 'procrastination', 'procrastinat

With a UMAP reduction first, for comparison. Less noise, but stochastic and not the calibrated configuration.

In [24]:
clusters_umap = clusterer(relations, umap=dict(n_components=5, n_neighbors=15, min_dist=0.0))

/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


cluster: 1,219 spans -> 299 clusters | 170 noise (14%) | umap {'n_components': 5, 'n_neighbors': 15, 'min_dist': 0.0}


/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


## 7. `standardize_groups` — groups to taxonomy concepts

```python
named = standardize_groups(groups, name_by="mean_sim", taxonomy="elsst", strategy="enriched", top=5, ...)
```

`mean_sim` averages the members' score vectors (conservative). `centroid` averages the vectors
first (scores higher). `medoid` scores the central member only. A low `mean_sim` score signals a
mixed group, or a construct the thesaurus lacks.

In [25]:
for rule in ("mean_sim", "centroid", "medoid"):
    named = standardize_groups(clusters, name_by=rule, top=3)
    multi = sorted(((k, v) for k, v in named.items() if v["size"] > 1), key=lambda kv: -kv[1]["size"])
    for k, v in multi[:4]:
        m = v["matches"][0]
        print(f"  {rule:<9} C{k} n={v['size']:<3} [{v['central']:<46}] -> {m['leaf']} ({m['score']:.2f})")
    print()
k, v = multi[0]
print("full record for one group:")
print({"central": v["central"], "size": v["size"], "members": v["members"][:4], "matches": v["matches"][:2]})

taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 724 groups named by mean_sim | best score: median 0.48, min 0.16, max 0.77
  mean_sim  C214 n=12  [higher intrinsic motivation                   ] -> INCENTIVES (0.58)
  mean_sim  C79 n=10  [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful] -> DISTANCE EDUCATION (0.56)
  mean_sim  C85 n=10  [perceived competence                          ] -> PERSONAL EFFICACY (0.52)
  mean_sim  C127 n=10  [procrastination                               ] -> SELF-DISCIPLINE (0.43)

taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 724 groups named by centroid | best score: median 0.49, min 0.16, max 0.80
  centroid  C214 n=12  [higher intrinsic motivation                   ] -> INCENTIVES (0.66)
  centroid  C79 n=10  [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful] -> DISTANCE EDUCATION (0.69)
  centroid  C85 n=10  [perceived competence                          ] -> PERSONAL EFFICACY (0.62)
  centroid  C127 n=10  [procrastination                               ] -> SELF-DISCIPLINE (0.49)

taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 724 groups named by medoid | best score: median 0.48, min 0.16, max 0.80
  medoid    C214 n=12  [higher intrinsic motivation                   ] -> INCENTIVES (0.65)
  medoid    C79 n=10  [distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful] -> DISTANCE EDUCATION (0.65)
  medoid    C85 n=10  [perceived competence                          ] -> PERSONAL EFFICACY (0.60)
  medoid    C127 n=10  [procrastination                               ] -> SELF-DISCIPLINE (0.50)

full record for one group:
{'central': 'higher intrinsic motivation', 'size': 12, 'members': ['greater classroom motivation', "learner's intrinsic motivation was", 'intrinsic motivation', 'high intrinsic motivation'], 'matches': [{'id': 'https://elsst.cessda.eu/id/5/5d1671fa-2940-4d42-b993-599b3ed84e76', 'leaf': 'INCENTIVES', 'path': 'MOTIVATION > INCENTIVES', 'score': 0.6484}, {'id': 'https://elsst.cessda.eu/id/5/d81ee447-fde7-4df6-931f-0e4f1f5bdcc7', 'le

### 7.3 `label_groups` — a label without a taxonomy

```python
labelled = label_groups(groups, by="both", top=3)     # medoid | keywords | both
```

No model, no taxonomy, no file. `medoid` is the `central` span the grouping already gives.
`keywords` ranks the content words of the member spans by class-based TF-IDF over the groups:
`(count in the group / words in the group) * log(1 + groups / groups that hold the word)`. The
first factor rewards a word the members repeat, the second punishes a word every group holds.
A group with one member takes the member as its label. The spans stay as they are.

In [26]:
for by in ("medoid", "keywords", "both"):
    labelled = label_groups(clusters, by=by, top=3)
    big = sorted(((k, v) for k, v in labelled.items() if v["size"] > 1), key=lambda kv: -kv[1]["size"])[:5]
    for k, v in big:
        print(f"  C{k} n={v['size']:<3} {v['name']}")
    print()
print("full record:", labelled[big[2][0]])
one = next(v for v in labelled.values() if v["size"] == 1)
print("a group of one:", one["name"])

labels: 724 groups labelled by medoid | 244 with 2 or more members
  C214 n=12  higher intrinsic motivation
  C79 n=10  distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful
  C85 n=10  perceived competence
  C127 n=10  procrastination
  C240 n=10  promoting learner autonomy

labels: 724 groups labelled by keywords | 244 with 2 or more members
  C214 n=12  motivation intrinsic learning
  C79 n=10  distance learning teaching
  C85 n=10  competence perceived students'perceptions
  C127 n=10  procrastination procrastinated procrastinate
  C240 n=10  autonomy learner promoting

labels: 724 groups labelled by both | 244 with 2 or more members
  C214 n=12  higher intrinsic motivation | motivation intrinsic learning
  C79 n=10  distance teaching requires ample preparation time and situation-specific didactic adaptation to be successful | distance learning teaching
  C85 n=10  perceived competence | competence perceived students'perceptio

### 7.4 The two labels side by side

Where the thesaurus holds the construct its label is the better one: comparable across
studies, and it carries a definition. Where the thesaurus is far, the score says so and the
keyword label is the honest one. Report both.

In [27]:
elsst = {k: v["matches"][0] for k, v in standardize_groups(clusters, name_by="mean_sim", top=1).items()}
labelled = label_groups(clusters, by="keywords", top=3)
print()
for k, v in sorted(((k, v) for k, v in labelled.items() if v["size"] > 1), key=lambda kv: -kv[1]["size"])[:6]:
    m = elsst[k]
    print(f"  C{k} n={v['size']:<3} {v['name']:<40} ELSST: {m['leaf']} ({m['score']:.2f})")

taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 724 groups named by mean_sim | best score: median 0.48, min 0.16, max 0.77
labels: 724 groups labelled by keywords | 244 with 2 or more members

  C214 n=12  motivation intrinsic learning            ELSST: INCENTIVES (0.58)
  C79 n=10  distance learning teaching               ELSST: DISTANCE EDUCATION (0.56)
  C85 n=10  competence perceived students'perceptions ELSST: PERSONAL EFFICACY (0.52)
  C127 n=10  procrastination procrastinated procrastinate ELSST: SELF-DISCIPLINE (0.43)
  C240 n=10  autonomy learner promoting               ELSST: LEARNING (0.46)
  C141 n=9   poor self-rated health                   ELSST: HEALTH STATUS (0.47)


The functions `label_groups` and `standardize_groups` are taken apart line by line in
`tutorials/PACKAGE_LINE_BY_LINE.ipynb`, Part 3.

The same function on the pairwise groups gives the same shape.

In [28]:
named_pairwise = standardize_groups(groups, name_by="mean_sim", top=3)
k = next(k for k, v in named_pairwise.items() if 3 <= v["size"] <= 8)
named_pairwise[k]

taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 669 groups named by mean_sim | best score: median 0.45, min 0.16, max 0.76


{'central': 'experimental intervention',
 'size': 4,
 'members': ['experimental intervention',
  'experimental interventions',
  'experimental treatment',
  'experimental intervention in this group'],
 'matches': [{'id': 'https://elsst.cessda.eu/id/5/9ecc4359-36f2-495a-a685-5bf729329efd',
   'leaf': 'PSYCHOLOGICAL TESTS',
   'path': 'RESEARCH > PSYCHOLOGICAL RESEARCH > PSYCHOLOGICAL TESTS',
   'score': 0.4444},
  {'id': 'https://elsst.cessda.eu/id/5/f4a93a9f-2c5c-404f-b582-e326099cbfd1',
   'leaf': 'DEMONSTRATIONS (LESSONS)',
   'path': 'EDUCATION > EDUCATIONAL METHODS > LESSONS > DEMONSTRATIONS (LESSONS)',
   'score': 0.3912},
  {'id': 'https://elsst.cessda.eu/id/5/482e42ce-7ce5-4a31-b7dd-632d6fdc7df8',
   'leaf': 'DRUG THERAPY',
   'path': 'MEDICAL SCIENCES > MEDICAL CARE > MEDICAL TREATMENT METHODS > DRUG THERAPY',
   'score': 0.3731}]}

## 8. Save what you want to keep

The functions return lists and dicts. `outputs/` is gitignored.

In [29]:
import csv
json.dump(matches, open("outputs/matches_enriched.json", "w"))
rows = [{"span": s, "group": g["group"], "central": g["central"]} for s, g in clusters.items()]
json.dump(rows, open("outputs/clusters.json", "w"))
with open("outputs/sentences_sample.csv", "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=data["sentences"][0].keys())
    w.writeheader(); w.writerows(data["sentences"])
print(sorted(os.listdir("outputs")))

['clusters.json', 'dois.csv', 'matches_enriched.json', 'relations_sample.json', 'sentences_sample.csv']


## 9. Traceability, end to end

```
doc_id ── page ── sent_id ── rel_id ── (cause span, effect span) ── span → node
```

The cell builds the edges of the pairwise groups the way the graph stage will: rewrite each
relation with the node of its cause and the node of its effect, merge identical pairs, count.
Then it walks the most supported edge back to its sentences.

In [30]:
node = {s: g["group"] for s, g in groups.items()}
central = {g["group"]: g["central"] for g in groups.values()}
edges = collections.defaultdict(list)
for r in relations:
    edges[(node[r["cause"]], node[r["effect"]])].append(r)
loops = sum(len(v) for (a, b), v in edges.items() if a == b)
print(f"{len(edges)} distinct edges from {len(relations):,} relations | {loops} relations became self-loops")
(a, b), support = max(((k, v) for k, v in edges.items() if k[0] != k[1]), key=lambda kv: len(kv[1]))
print()
print(f"edge  [{central[a]}] -> [{central[b]}]   weight {len(support)}   pages {sorted({r['page'] for r in support})}")
for r in support[:3]:
    print(f"  {r['rel_id']}  p{r['page']}  {r['cause']!r} -> {r['effect']!r}")
    print(f"      \"{r['sentence']}\"")

758 distinct edges from 979 relations | 79 relations became self-loops

edge  [high social support from students, teachers, and family] -> [lower levels of loneliness]   weight 19   pages [1, 2, 4, 11, 13, 16, 17]
  d005p0002s012r00  p1  'high social support from family, teachers, and students' -> 'loneliness'
      "High social support from family, teachers, and students was associated with reduced loneliness and better mental health outcomes."
  d005p0005s000r00  p2  'targeted interventions to promote social support in schools' -> 'address loneliness'
      "Schütz et al. BMC Public Health 25:2172 Page 2 of 21 mental health, so targeted interventions to promote social support in schools and families are needed to address loneliness."
  d005p0015s001r00  p4  'social support' -> 'individuals perceive feelings of loneliness as less stressful and more changeable'
      "Social support may help individuals perceive feelings of loneliness as less stressful and more changeable, thus bufferi

## 10. A full run on the sample papers

This section runs the whole pipeline once, on all PDF files in `sample_files/`. Every cell runs for
real. No cell loads a saved file. The sections above show one function at a time. This section
shows how the results of all functions fit together.

The steps:

1. `harvest` reads the folder.
2. `extract_dois` finds the DOI of each paper and saves `outputs/dois_full_run.csv`.
3. `causenet` finds the cause-effect pairs. On CPU this takes about 2 minutes.
4. The harmonizer groups the spans and names the groups: with ELSST, and without a thesaurus.
5. The last cells show the results per paper with its DOI, the largest groups, and the strongest edges.

### 10.1 Harvest the folder

In [4]:
full = harvest("sample_files")

  reading .pdf file 5 of 7

  reading .pdf file 7 of 7
harvest: sample_files
  files: 8 | read: 7 | skipped: 1
    skipped: SOURCES.md: the file name does not end with .pdf (not_pdf)
  text: 2,547 sentences kept | 1,849 rows dropped
       423  table_row
       328  no_lowercase
       247  reference_section
       229  too_short
       148  low_alpha
       107  running_head
       106  heading
        74  digit_heavy
        72  spaced_letters
        52  caption_or_keywords
        31  ref_fragment
        13  index_entry
         8  doi_url
         7  too_long
         2  empty_page
         1  page_number
         1  front_back_matter
  sentences per file:
       265  alrabai_2021_autonomy_supportive_teaching
       297  barrech_2018_job_insecurity_health
       301  friesinger_2025_discrimination_health
       385  gordesli_2024_social_media_mental_health
       318  pelikan_2021_needs_intrinsic_motivation
       616  schutz_2025_loneliness_social_support
       365  stephany_2017_income_inequality_trust


### 10.1b The DOI of each paper

The DOI links each `doc_id` to the published paper. Section 10.4 shows it next to the counts.

In [5]:
full_dois = extract_dois("sample_files", save="outputs/dois_full_run.csv")
doi_of = {}
for row in full_dois:
    doi_of[row["doc_id"]] = row["doi"]

  reading .pdf file 7 of 7
dois: sample_files
  files: 8 | found: 7 | not found: 0 | skipped: 1
    not_pdf: SOURCES.md: the file name does not end with .pdf
  saved: outputs/dois_full_run.csv


### 10.2 Find the cause-effect pairs

In [32]:
full_relations = causenet(full["sentences"])

  64/2,547 sentences (3s)

  128/2,547 sentences (6s)

  192/2,547 sentences (10s)

  256/2,547 sentences (13s)

  320/2,547 sentences (16s)

...

  2,304/2,547 sentences (113s)

  2,368/2,547 sentences (117s)

  2,432/2,547 sentences (119s)

  2,496/2,547 sentences (122s)

  2,547/2,547 sentences (123s)
causenet: 481 causal sentences of 2,547 | 979 relations
  spans repaired after decoding: 124 | both spans found verbatim in their sentence: 97%


### 10.3 Group and name the spans

Four calls. Each one takes the output of the call before it, or the relations.

In [33]:
full_matches  = standardize_constructs(full_relations, taxonomy="elsst", leaves_only=False, top=5)
full_groups   = pairwise_grouping(full_relations, threshold=0.69)
full_clusters = clusterer(full_relations)
full_named    = standardize_groups(full_clusters, name_by="mean_sim", top=5)
full_labelled = label_groups(full_clusters, by="both", top=3)

taxonomy: embeddings from cache (elsst_enriched_allmpnet_all.npy)


standardize: 1,219 spans -> top 5 of 3,739 entries | best score: median 0.51, min 0.16, max 0.83


pairwise: 1,219 spans -> 669 groups at cos >= 0.69 | 90 groups with 2+ members, largest 153


/home/rass/Desktop/projects/theoryminer/.venv/lib/python3.13/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


cluster: 1,219 spans -> 244 clusters | 480 noise (39%) | raw embeddings
taxonomy: embeddings from cache (elsst_enriched_allmpnet_leaves.npy)


standardize: 724 groups named by mean_sim | best score: median 0.48, min 0.16, max 0.77
labels: 724 groups labelled by both | 244 with 2 or more members


### 10.4 Results per paper

The DOI, the count of kept sentences and the count of relations for each paper. A paper with more causal claims in
its text gives more relations per sentence.

In [7]:
sentences_per_doc = collections.Counter(r["doc_id"] for r in full["sentences"])
relations_per_doc = collections.Counter(r["doc_id"] for r in full_relations)

print(f"{'paper':<44} {'DOI':<30} {'sentences':>9} {'relations':>9}")
for doc_id in sorted(sentences_per_doc):
    print(f"{doc_id:<44} {doi_of.get(doc_id, ''):<30} {sentences_per_doc[doc_id]:>9} {relations_per_doc[doc_id]:>9}")
print(f"{'all':<44} {'':<30} {len(full['sentences']):>9} {len(full_relations):>9}")

paper                                        DOI                            sentences relations
alrabai_2021_autonomy_supportive_teaching    10.3389/fpsyg.2021.728657            265       154
barrech_2018_job_insecurity_health           10.1186/s12889-018-5621-4            297        98
friesinger_2025_discrimination_health        10.1186/s12889-025-23888-6           301       116
gordesli_2024_social_media_mental_health     10.1371/journal.pone.0316365         385       218
pelikan_2021_needs_intrinsic_motivation      10.1371/journal.pone.0257346         318       121
schutz_2025_loneliness_social_support        10.1186/s12889-025-23247-5           616       202
stephany_2017_income_inequality_trust        10.1007/s11205-016-1460-9            365        70
all                                                                              2547       979


Three relations from each paper, with the `rel_id` and the page:

In [35]:
for doc_id in sorted(sentences_per_doc):
    print(doc_id)
    doc_relations = [r for r in full_relations if r["doc_id"] == doc_id]
    for r in doc_relations[:3]:
        print(f"   [{r['rel_id']} p{r['page']}] {r['cause']!r} -> {r['effect']!r}")

alrabai_2021_autonomy_supportive_teaching
   [d000p0003s003r00 p1] 'experimental intervention' -> 'efl ehsan namaziandost'
   [d000p0006s003r00 p1] 'desire' -> 'autonomy'
   [d000p0006s003r01 p1] 'desire' -> 'take charge'
barrech_2018_job_insecurity_health
   [d001p0004s001r00 p1] 'changing economic environment' -> 'organizational change'
   [d001p0004s001r01 p1] 'changing economic environment' -> 'reorganizations'
   [d001p0004s001r02 p1] 'changing economic environment' -> 'working environments'
friesinger_2025_discrimination_health
   [d002p0002s000r00 p1] 'factors' -> 'experience discrimination'
   [d002p0002s000r01 p1] 'age, gender, illness' -> 'experience discrimination'
   [d002p0002s000r02 p1] 'disability' -> 'experience discrimination'
gordesli_2024_social_media_mental_health
   [d003p0006s002r00 p1] 'on social media' -> 'social media use influences the mental funding'
   [d003p0008s001r00 p1] 'diversity of social media activities' -> 'complicates this issue'
   [d003p0008s001r

### 10.5 The largest groups, with two names

`label_groups` names a cluster from its own words. `standardize_groups` names it with the closest
ELSST concept and a score. A low score means that ELSST has no good concept for the group.

In [36]:
multi = [(k, v) for k, v in full_labelled.items() if v["size"] > 1]
multi.sort(key=lambda kv: -kv[1]["size"])

print(f"{len(multi)} clusters with 2 or more spans; the 10 largest:")
for k, v in multi[:10]:
    best = full_named[k]["matches"][0]
    print(f"  C{k:<4} n={v['size']:<3} {v['name'][:58]:<58} ELSST: {best['leaf']} ({best['score']:.2f})")

244 clusters with 2 or more spans; the 10 largest:
  C214  n=12  higher intrinsic motivation | motivation intrinsic learnin ELSST: INCENTIVES (0.58)
  C79   n=10  distance teaching requires ample preparation time and situ ELSST: DISTANCE EDUCATION (0.56)
  C85   n=10  perceived competence | competence perceived students'perce ELSST: PERSONAL EFFICACY (0.52)
  C127  n=10  procrastination | procrastination procrastinated procrasti ELSST: SELF-DISCIPLINE (0.43)
  C240  n=10  promoting learner autonomy | autonomy learner promoting    ELSST: LEARNING (0.46)
  C141  n=9   poor self-rated health | poor self-rated health            ELSST: HEALTH STATUS (0.47)
  C70   n=8   prevented some from participating in the study | study cha ELSST: SOCIO-MEDICAL INDICES (0.34)
  C200  n=7   our level of generalized trust | trust generalized level   ELSST: INTERPERSONAL TRUST (0.62)
  C233  n=7   promote efl learner autonomy | efl learner autonomy        ELSST: EDUCATIONAL EXPECTATIONS (0.40)
  C17   n=6 

### 10.6 The strongest edges

The graph stage is not written yet. This cell shows what it will do. Each span becomes the label
of its cluster. A noise span keeps its own text. Each relation then becomes an edge from the cause
node to the effect node. The weight of an edge is the number of relations behind it.

In [37]:
def node_of(span):
    """Return the node label of a span: its cluster label, or the span for a noise span."""
    group = full_clusters[span]["group"]
    if group == -1:
        return full_labelled[span]["name"]
    return full_labelled[group]["name"]


edges = collections.defaultdict(list)
for r in full_relations:
    cause_node = node_of(r["cause"])
    effect_node = node_of(r["effect"])
    if cause_node != effect_node:
        edges[(cause_node, effect_node)].append(r)

strongest = sorted(edges.items(), key=lambda kv: -len(kv[1]))
print(f"{len(edges):,} distinct edges from {len(full_relations):,} relations; the 8 strongest:")
for (cause_node, effect_node), support in strongest[:8]:
    papers = sorted({r["doc_id"].split("_")[0] for r in support})
    print(f"  weight {len(support):<3} [{cause_node[:40]}] -> [{effect_node[:40]}]   papers: {papers}")

906 distinct edges from 979 relations; the 8 strongest:
  weight 4   [racism] -> [health inequities | inequities health pr]   papers: ['friesinger']
  weight 4   [higher intrinsic motivation | motivation] -> [procrastination | procrastination procra]   papers: ['pelikan']
  weight 3   [job insecurity | insecurity job uncertai] -> [unhealthy employees | employees health t]   papers: ['barrech']
  weight 3   [scored low to medium for horizontal indi] -> [levels of depression | depression levels]   papers: ['gordesli']
  weight 3   [perceived competence | competence percei] -> [higher intrinsic motivation | motivation]   papers: ['pelikan']
  weight 3   [loneliness] -> [multiple psychosomatic complaints | psyc]   papers: ['schutz']
  weight 3   [loneliness] -> [mental health | mental health problems]   papers: ['schutz']
  weight 2   [implementation of autonomy-promoting str] -> [promoting learner autonomy | autonomy le]   papers: ['alrabai']


One edge, walked back to its sentences:

In [38]:
(cause_node, effect_node), support = strongest[0]
print(f"edge [{cause_node}] -> [{effect_node}], weight {len(support)}")
for r in support[:3]:
    print(f"  {r['rel_id']}  {r['doc_id']}  p{r['page']}")
    print(f"     {r['cause']!r} -> {r['effect']!r}")
    print(f"     \"{r['sentence']}\"")

edge [racism] -> [health inequities | inequities health produces], weight 4
  d002p0006s014r00  friesinger_2025_discrimination_health  p2
     'racism' -> 'societal inequities'
     "The consequences of racism include societal inequities and adverse health outcomes, such as poorer physical and mental health [13, 19]."
  d002p0007s003r00  friesinger_2025_discrimination_health  p2
     'racism' -> 'produces ethnic health inequalities'
     "With this epidemiological lens, racism produces ethnic health inequalities that manifest as health inequities through specific pathways [3, 9]."
  d002p0007s003r01  friesinger_2025_discrimination_health  p2
     'racism' -> 'health inequities'
     "With this epidemiological lens, racism produces ethnic health inequalities that manifest as health inequities through specific pathways [3, 9]."


### 10.7 Checks after the run

Four numbers to read after every run:

- the spans in both roles: a chain in the graph passes only through these;
- the largest pairwise group: the chaining check (section 5.1);
- the share of spans with an ELSST score at or above 0.55;
- the noise share of `clusterer`.

In [39]:
causes = {r["cause"] for r in full_relations}
effects = {r["effect"] for r in full_relations}
pairwise_sizes = collections.Counter(g["group"] for g in full_groups.values())
at_tau = sum(m[0]["score"] >= 0.55 for m in full_matches.values()) / len(full_matches)
noise = sum(g["group"] == -1 for g in full_clusters.values()) / len(full_clusters)

print("spans in both roles       :", len(causes & effects))
print("largest pairwise group    :", max(pairwise_sizes.values()))
print(f"ELSST score >= 0.55       : {at_tau:.0%}")
print(f"noise share of clusterer  : {noise:.0%}")

spans in both roles       : 47
largest pairwise group    : 153
ELSST score >= 0.55       : 35%
noise share of clusterer  : 39%


## 11. Settings at a glance

| function | setting | default | source of the default |
|---|---|---|---|
| `harvest` | `min_words` / `max_words` | 4 / 80 | old cleaning notebook |
| `extract_dois` | `first_pages` / `save` | 2 / `None` | the paper's own DOI is on page 1 (checked by hand: 7 of 7) |
| `causenet` | `threshold` | 0.8 | old pipeline sweep 0.5–0.8 |
| `causenet` | `mode` / `decision` | `neural` / `cls+span` | model card defaults |
| `causenet` | `batch_size` / `device` | 64 / `auto` | `auto` = an NVIDIA GPU, else an Apple GPU, else the CPU |
| all harmonizer | `embeddings` | `allmpnet` | Paper 3; `bge_base` scores 0.02 F1 higher (`STRATEGIES.md` section 5) |
| `standardize_*` | `taxonomy` | `elsst` | project choice; the only built-in thesaurus |
| `standardize_*` | `strategy` | `enriched` | Paper 3 web tool; confirmed by the held-out test |
| `standardize_*` | `top` | 5 | project choice |
| `standardize_*` | `leaves_only` | `True` | a granularity choice; `False` covers broad SDT constructs |
| `standardize_groups` | `name_by` | `mean_sim` | old pipeline's cluster-naming comparison |
| `pairwise_grouping` | `threshold` | 0.69 | Paper 3 `BEST_PAIRWISE_THRESHOLDS` |
| `clusterer` | `min_cluster_size` / `min_samples` | 2 / 2 | Paper 3 `BEST_CLUSTERING_PARAMS` |
| `clusterer` | `umap` | `None` | Paper 3; the best run had no reduction |

τ is not a setting of any function. Apply it downstream, per model, per strategy, per thesaurus.
The evidence behind the defaults is in `tutorials/STRATEGIES.md` section 5.